# MT Evaluation — Error Analysis
**Input:** `results_raw.xlsx` produced by `testing_on_indicMT_data.py`

Sections:
1. Overall Recall, Precision & F1
2. Score Correlation — agent score vs Human_score
3. Per-Category Precision, Recall, F1
4. Error Type × Severity Heatmap
5. Confusion Matrix — gold vs predicted
6. False Positive & False Negative breakdown
7. F1 Distribution
8. Export to Excel

## 0. Imports & Config

In [ ]:
import json
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')

RESULTS_PATH          = 'results_raw.xlsx'
PRED_THRESHOLD        = 0.35
HIGH_PRECISION_THRESH = 0.5
HIGH_RECALL_THRESH    = 0.5
F1_GOOD_THRESH        = 0.5

SUPER_CATEGORIES = {
    'Semantic Equivalent':         ['Identical', 'Word_synm', 'Fluent', 'Mixd_lang', 'Negt_anto', 'Default_similar'],
    'Semantic Contradiction':      ['Anto_flip', 'Negt_flip', 'Word_rplc', 'Neutral', 'Default_dissimilar'],
    'Grammatical / Morphological': ['Gend_flip', 'Sing_plul', 'Tens_chng', 'Word_ordr'],
    'Information Completeness':    ['Add_extra', 'Omission'],
}
ALL_CATS   = [c for cats in SUPER_CATEGORIES.values() for c in cats]
CAT_TO_SUP = {c: s for s, cats in SUPER_CATEGORIES.items() for c in cats}

SUPER_COLORS = {
    'Semantic Equivalent':         '#1f77b4',
    'Semantic Contradiction':      '#d62728',
    'Grammatical / Morphological': '#2ca02c',
    'Information Completeness':    '#ff7f0e',
}

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f7f7f7',
    'axes.edgecolor':   '#cccccc',
    'axes.labelcolor':  '#222222',
    'xtick.color':      '#444444',
    'ytick.color':      '#444444',
    'text.color':       '#222222',
    'grid.color':       '#dddddd',
    'grid.linewidth':   0.7,
    'font.family':      'sans-serif',
    'axes.titlecolor':  '#111111',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})

print('Config loaded.')

## 1. Load & Parse Results

In [ ]:
df_raw = pd.read_excel(RESULTS_PATH, engine='openpyxl')

if 'run_status' in df_raw.columns:
    df = df_raw[df_raw['run_status'] == 'ok'].copy()
else:
    df = df_raw.copy()

print(f'Total rows loaded : {len(df_raw)}')
print(f'Rows used         : {len(df)}')
print(f'Columns           : {list(df.columns)}')

def parse_json(v):
    if pd.isna(v) or str(v).strip() == '': return []
    try: return json.loads(v)
    except: return []

df['gold_list'] = df['gold_cats'].apply(parse_json)
df['pred_list'] = df['agent_predicted_cats'].apply(parse_json)
df['has_gold']  = df['gold_list'].apply(lambda x: len(x) > 0)

def row_metrics(row):
    gold = set(row['gold_list'])
    pred = set(row['pred_list'])
    hits = gold & pred
    p  = len(hits) / len(pred) if pred else 0.0
    r  = len(hits) / len(gold) if gold else None
    f1 = (2*p*r/(p+r)) if (r is not None and (p+r) > 0) else (None if r is None else 0.0)
    return pd.Series({'P': p, 'R': r, 'F1': f1})

metrics  = df.apply(row_metrics, axis=1)
df['P']  = metrics['P']
df['R']  = metrics['R']
df['F1'] = metrics['F1']

valid = df[df['has_gold']].copy()
print(f'Rows with gold annotations: {len(valid)}')

## 2. Overall Recall, Precision & F1

In [ ]:
mean_p  = valid['P'].mean()
mean_r  = valid['R'].dropna().mean()
mean_f1 = valid['F1'].dropna().mean()
n_total = len(df)
n_valid = len(valid)

n_f1_good   = int((valid['F1'].dropna() >= F1_GOOD_THRESH).sum())
pct_f1_good = n_f1_good / n_total * 100
n_hi_prec   = int((valid['P'] >= HIGH_PRECISION_THRESH).sum())
pct_hi_prec = n_hi_prec / n_total * 100
n_hi_rec    = int((valid['R'].dropna() >= HIGH_RECALL_THRESH).sum())
pct_hi_rec  = n_hi_rec / n_total * 100

summary = pd.DataFrame([
    ('Total Samples',                                        n_total,      ''),
    ('Rows with Gold Annotations',                           n_valid,      ''),
    ('Mean Recall',                                          f'{mean_r:.4f}',  f'{mean_r*100:.2f}%'),
    ('Mean Precision',                                       f'{mean_p:.4f}',  f'{mean_p*100:.2f}%'),
    ('Mean F1',                                              f'{mean_f1:.4f}', f'{mean_f1*100:.2f}%'),
    (f'Predictions with F1 >= {F1_GOOD_THRESH}',             n_f1_good,    f'{pct_f1_good:.1f}% of total'),
    (f'High Precision Cases (P >= {HIGH_PRECISION_THRESH})', n_hi_prec,    f'{pct_hi_prec:.1f}% of total'),
    (f'High Recall Cases (R >= {HIGH_RECALL_THRESH})',        n_hi_rec,    f'{pct_hi_rec:.1f}% of total'),
], columns=['Metric', 'Value', 'Percentage'])

display(summary.style.set_properties(**{'text-align': 'left'}))

fig, ax = plt.subplots(figsize=(8, 3))
labels = ['Recall', 'Precision', 'F1']
values = [mean_r, mean_p, mean_f1]
colors = ['#1f77b4', '#d62728', '#2ca02c']
bars   = ax.barh(labels, values, color=colors, height=0.45)
for bar, val in zip(bars, values):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val*100:.2f}%', va='center', fontsize=11, color='#111111')
ax.set_xlim(0, 1.15)
ax.set_xlabel('Score')
ax.set_title('Overall Recall / Precision / F1')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.grid(axis='x', alpha=0.5)
plt.tight_layout()
plt.show()

## 3. Score Correlation — Agent Score vs Human Score

In [ ]:
score_df = df[['agent_quality_score', 'Human_score']].dropna()

if len(score_df) >= 3:
    pearson_r,  pearson_p  = stats.pearsonr(score_df['agent_quality_score'], score_df['Human_score'])
    spearman_r, spearman_p = stats.spearmanr(score_df['agent_quality_score'], score_df['Human_score'])

    print(f'Rows with both scores : {len(score_df)}')
    print(f'Pearson  r   = {pearson_r:.4f}   p = {pearson_p:.4f}')
    print(f'Spearman rho = {spearman_r:.4f}   p = {spearman_p:.4f}')

    delta = score_df['agent_quality_score'] - score_df['Human_score']

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(delta, bins=30, color='#1f77b4', edgecolor='white', alpha=0.85)
    ax.axvline(0,            color='#ff7f0e', linewidth=1.8, linestyle='--', label='zero delta')
    ax.axvline(delta.mean(), color='#d62728', linewidth=1.8, linestyle='-',
               label=f'mean delta = {delta.mean():.2f}')
    ax.set_xlabel('Agent Score - Human Score')
    ax.set_ylabel('Count')
    ax.set_title('Distribution of Score Delta (Agent - Human)')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.4)
    plt.tight_layout()
    plt.show()
else:
    print('Not enough rows with both agent_quality_score and Human_score to compute correlation.')

## 4. Per-Category Precision, Recall, F1

In [ ]:
tp = Counter()
fp = Counter()
fn = Counter()

for _, row in df.iterrows():
    gold = set(row['gold_list'])
    pred = set(row['pred_list'])
    for c in pred & gold: tp[c] += 1
    for c in pred - gold: fp[c] += 1
    for c in gold - pred: fn[c] += 1

records = []
for cat in ALL_CATS:
    t, fp_, fn_ = tp[cat], fp[cat], fn[cat]
    support = t + fn_
    p  = t / (t + fp_) if (t + fp_) else 0.0
    r  = t / (t + fn_) if (t + fn_) else 0.0
    f1 = 2*p*r/(p+r) if (p+r) else 0.0
    records.append({'Super': CAT_TO_SUP[cat], 'Category': cat,
                    'TP': t, 'FP': fp_, 'FN': fn_, 'Support': support,
                    'Precision': round(p,4), 'Recall': round(r,4), 'F1': round(f1,4)})

cat_df = pd.DataFrame(records)
display(cat_df.sort_values('F1', ascending=False).reset_index(drop=True))

fig, ax = plt.subplots(figsize=(16, 6))
x = np.arange(len(ALL_CATS))
w = 0.26
ax.bar(x - w, cat_df['Precision'], w, label='Precision', color='#1f77b4', alpha=0.88)
ax.bar(x,     cat_df['Recall'],    w, label='Recall',    color='#d62728', alpha=0.88)
ax.bar(x + w, cat_df['F1'],        w, label='F1',        color='#2ca02c', alpha=0.88)
ax.set_xticks(x)
ax.set_xticklabels(ALL_CATS, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.15)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend(fontsize=10)
ax.set_title('Per Sub-Category Precision / Recall / F1')
ax.grid(axis='y', alpha=0.4)
for tick, cat in zip(ax.get_xticklabels(), ALL_CATS):
    tick.set_color(SUPER_COLORS[CAT_TO_SUP[cat]])
plt.tight_layout()
plt.show()

sc_avg = cat_df.groupby('Super')[['Precision','Recall','F1']].mean().round(4)
print('\nPer Super-Category Averages')
display(sc_avg)

## 5. Error Type x Severity Heatmap

In [ ]:
sev_records = []
for _, row in df.iterrows():
    for i in range(1, 4):
        t_col = f'Own_Cat{i}_Type'
        s_col = f'Own_Cat{i}_Severity'
        if t_col not in df.columns or s_col not in df.columns:
            continue
        t = str(row.get(t_col, '')).strip()
        s = str(row.get(s_col, '')).strip()
        if t.lower() in {'nan', 'none', '', 'n/a', 'default', 'other', 'source_error'}:
            continue
        if s.lower() in {'nan', 'none', ''}:
            s = 'Unknown'
        sev_records.append({'ErrorType': t, 'Severity': s})

if sev_records:
    sev_df = pd.DataFrame(sev_records)
    pivot  = sev_df.groupby(['ErrorType', 'Severity']).size().unstack(fill_value=0)
    preferred_order = ['Very Low', 'Low', 'Default', 'Medium', 'High', 'Very High', 'Unknown']
    pivot = pivot.reindex(
        columns=[c for c in preferred_order if c in pivot.columns] +
                [c for c in pivot.columns if c not in preferred_order]
    )
    fig, ax = plt.subplots(figsize=(max(10, len(pivot.columns)*1.6), max(6, len(pivot)*0.55)))
    sns.heatmap(
        pivot, annot=True, fmt='d', cmap='YlOrRd',
        linewidths=0.4, linecolor='white',
        ax=ax, cbar_kws={'shrink': 0.7}
    )
    ax.set_title('Error Types vs Severity Levels')
    ax.set_xlabel('Severity', fontsize=11)
    ax.set_ylabel('Error Type', fontsize=11)
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)
    plt.tight_layout()
    plt.show()
    print(f'Total annotations in heatmap: {len(sev_df)}')
else:
    print('No Own_Cat*_Type / Own_Cat*_Severity columns found — skipping heatmap.')

## 6. Confusion Matrix — Gold vs Predicted

In [ ]:
cooc = defaultdict(Counter)
for _, row in df.iterrows():
    gold = set(row['gold_list'])
    pred = set(row['pred_list'])
    for g in gold:
        for p in pred:
            cooc[g][p] += 1
        if not (gold & pred):
            cooc[g]['[MISSED]'] += 1

cols   = ALL_CATS + ['[MISSED]']
matrix = pd.DataFrame(0, index=ALL_CATS, columns=cols)
for g in ALL_CATS:
    for c, v in cooc[g].items():
        if c in matrix.columns:
            matrix.loc[g, c] = v

matrix = matrix.loc[(matrix.sum(axis=1) > 0), (matrix.sum(axis=0) > 0)]

if not matrix.empty:
    fig, ax = plt.subplots(figsize=(max(12, len(matrix.columns)*0.7),
                                    max(8,  len(matrix)*0.55)))
    sns.heatmap(
        matrix, annot=True, fmt='d', cmap='Blues',
        linewidths=0.3, linecolor='white',
        ax=ax, cbar_kws={'shrink': 0.6}
    )
    ax.set_title('Confusion Matrix: Gold (rows) vs Predicted (cols)')
    ax.set_xlabel('Predicted Category', fontsize=11)
    ax.set_ylabel('Gold Category',      fontsize=11)
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print('Confusion matrix is empty — no gold/predicted category overlap found.')

## 7. False Positive & False Negative Analysis

In [ ]:
def get_cat_score(row, cat):
    try:
        scores = json.loads(row.get('all_agent_scores_json', '[]'))
        for e in scores:
            if e.get('cat') == cat:
                return round(e.get('score', 0.0), 4)
    except: pass
    return float('nan')

fp_records, fn_records = [], []
for _, row in df.iterrows():
    gold = set(row['gold_list'])
    pred = set(row['pred_list'])
    for c in sorted(pred - gold):
        fp_records.append({'Category': c, 'Super': CAT_TO_SUP.get(c,''),
                           'Agent_Score': get_cat_score(row, c),
                           'Gold_Cats': row['gold_cats'],
                           'Source': str(row.get('Source',''))[:100]})
    for c in sorted(gold - pred):
        fn_records.append({'Category': c, 'Super': CAT_TO_SUP.get(c,''),
                           'Agent_Score': get_cat_score(row, c),
                           'Pred_Cats': row['agent_predicted_cats'],
                           'Source': str(row.get('Source',''))[:100]})

fp_df   = pd.DataFrame(fp_records) if fp_records else pd.DataFrame(columns=['Category','Super','Agent_Score','Gold_Cats','Source'])
fn_df   = pd.DataFrame(fn_records) if fn_records else pd.DataFrame(columns=['Category','Super','Agent_Score','Pred_Cats','Source'])
fp_freq = fp_df['Category'].value_counts() if not fp_df.empty else pd.Series(dtype=int)
fn_freq = fn_df['Category'].value_counts() if not fn_df.empty else pd.Series(dtype=int)

print(f'Total False Positives : {len(fp_df)}')
print(f'Total False Negatives : {len(fn_df)}')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, freq, title, color in [
    (axes[0], fp_freq, 'False Positives by Category\n(agent predicted, gold did not)', '#d62728'),
    (axes[1], fn_freq, 'False Negatives by Category\n(gold had, agent missed)',        '#ff7f0e'),
]:
    if freq.empty:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
    else:
        freq.plot(kind='barh', ax=ax, color=color, alpha=0.85)
        ax.set_xlabel('Count')
        ax.grid(axis='x', alpha=0.4)
        for p in ax.patches:
            ax.text(p.get_width() + 0.2, p.get_y() + p.get_height()/2,
                    str(int(p.get_width())), va='center', fontsize=9, color='#222222')
    ax.set_title(title)

plt.tight_layout()
plt.show()

print('\nTop 10 False Positive categories')
display(fp_freq.head(10).to_frame('FP_Count'))

print('\nTop 10 False Negative categories')
display(fn_freq.head(10).to_frame('FN_Count'))

## 8. F1 Distribution

In [ ]:
f1_vals = valid['F1'].dropna()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(f1_vals, bins=20, color='#1f77b4', edgecolor='white', alpha=0.85)
ax.axvline(F1_GOOD_THRESH, color='#ff7f0e', linewidth=1.8, linestyle='--',
           label=f'F1 = {F1_GOOD_THRESH} threshold')
ax.axvline(f1_vals.mean(), color='#d62728', linewidth=1.8, linestyle='-',
           label=f'mean F1 = {f1_vals.mean():.3f}')
ax.set_xlabel('F1 Score')
ax.set_ylabel('Number of Samples')
ax.set_title('F1 Score Distribution')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

print(f'Samples with F1 >= {F1_GOOD_THRESH} : {n_f1_good}  ({pct_f1_good:.1f}% of {n_total} total)')
print(f'High Precision (>= {HIGH_PRECISION_THRESH})       : {n_hi_prec}  ({pct_hi_prec:.1f}%)')
print(f'High Recall    (>= {HIGH_RECALL_THRESH})           : {n_hi_rec}  ({pct_hi_rec:.1f}%)')

## 9. Export Full Analysis to Excel

In [ ]:
OUTPUT_XLSX = 'error_analysis_results.xlsx'

with pd.ExcelWriter(OUTPUT_XLSX, engine='openpyxl') as writer:
    summary.to_excel(writer, sheet_name='Summary',      index=False)
    cat_df.to_excel(writer,  sheet_name='Per_Category', index=False)
    matrix.reset_index().to_excel(writer, sheet_name='Confusion', index=False)
    fp_df.to_excel(writer,   sheet_name='FP_Analysis',  index=False)
    fn_df.to_excel(writer,   sheet_name='FN_Analysis',  index=False)

    if sev_records:
        pivot.reset_index().to_excel(writer, sheet_name='ErrorType_x_Severity', index=False)

    sc_cols = [c for c in ['Source','Human_score','agent_quality_score','P','R','F1'] if c in df.columns]
    sc_out  = df[sc_cols].copy()
    sc_out['score_delta'] = sc_out['agent_quality_score'] - sc_out['Human_score']
    sc_out['abs_error']   = sc_out['score_delta'].abs()
    sc_out.sort_values('abs_error', ascending=False).to_excel(
        writer, sheet_name='Score_Comparison', index=False)

print(f'Analysis exported to: {OUTPUT_XLSX}')